# Supabase Crime Visualizations (Seaborn)

Fetch Chicago crime data from Supabase and generate report-friendly static charts with Seaborn/Matplotlib.

**Prereqs**: `pip install supabase python-dotenv pandas seaborn matplotlib` and ensure `.env` has `SUPABASE_URL` and `SUPABASE_KEY`.


In [ ]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

table_name = "chicago_crimes"  # Supabase table with aggregated crime stats
BATCH_SIZE = 1000               # adjust if you want bigger pages
LIMIT = None                    # set to an int to cap rows for quicker plotting


In [ ]:
from typing import Optional

def fetch_table(table: str, batch_size: int = 1000, limit: Optional[int] = None) -> pd.DataFrame:
    all_rows = []
    start = 0
    while True:
        end = start + batch_size - 1
        resp = supabase.table(table).select("*").range(start, end).execute()
        rows = resp.data or []
        all_rows.extend(rows)
        if len(rows) < batch_size or (limit and len(all_rows) >= limit):
            break
        start += batch_size
    if limit:
        all_rows = all_rows[:limit]
    return pd.DataFrame(all_rows)


df = fetch_table(table_name, batch_size=BATCH_SIZE, limit=LIMIT)
print(f"Fetched {len(df)} rows from '{table_name}'")
df.head()


In [ ]:
# Basic dtype fixes for plotting
if not df.empty:
    df["date_only"] = pd.to_datetime(df["date_only"], errors="coerce").dt.date
    df["crime_count"] = pd.to_numeric(df.get("crime_count", 0), errors="coerce").fillna(0).astype(int)

print(df.dtypes)
df.head()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

if df.empty:
    raise SystemExit("No data returned from Supabase. Check table name or credentials.")

sns.set_theme(style="whitegrid")

# Top crime types
primary_counts = (
    df.groupby("primary_type")["crime_count"].sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
plt.figure(figsize=(10, 5))
sns.barplot(data=primary_counts, x="primary_type", y="crime_count", color="#4C72B0")
plt.title("Top 10 Crime Types")
plt.xlabel("Crime Type")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Daily trend
by_day = (
    df.groupby("date_only")["crime_count"].sum()
    .reset_index()
    .sort_values("date_only")
)
plt.figure(figsize=(12, 4))
sns.lineplot(data=by_day, x="date_only", y="crime_count", color="#55A868")
plt.title("Crimes per Day")
plt.xlabel("Date")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Top locations
loc_counts = (
    df.groupby("location_description")["crime_count"].sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
plt.figure(figsize=(8, 6))
sns.barplot(data=loc_counts, x="crime_count", y="location_description", color="#C44E52")
plt.title("Top 10 Locations")
plt.xlabel("Count")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

# Arrest rate by crime type (show highest and lowest)
summary = (
    df.groupby(["primary_type", "arrest"])["crime_count"].sum()
    .reset_index()
    .pivot(index="primary_type", columns="arrest", values="crime_count")
    .fillna(0)
)
summary.columns = ["no_arrest", "arrest"] if True in summary.columns else summary.columns
summary["total"] = summary.sum(axis=1)
summary = summary[summary["total"] > 0]
summary["arrest_rate"] = summary.get("arrest", 0) / summary["total"]
summary = summary.sort_values("arrest_rate", ascending=False)

MIN_TOTAL = 100  # filter out very rare categories to avoid noisy rates; adjust as needed
filtered = summary[summary["total"] >= MIN_TOTAL]

highest = filtered.head(10)
lowest = filtered.tail(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=highest["arrest_rate"], y=highest.index, color="#8172B2")
plt.title("Highest Arrest Rates by Crime Type")
plt.xlabel("Arrest Rate")
plt.ylabel("Crime Type")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(x=lowest["arrest_rate"], y=lowest.index, color="#CCB974")
plt.title("Lowest Arrest Rates by Crime Type")
plt.xlabel("Arrest Rate")
plt.ylabel("Crime Type")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

print("Top 10 highest arrest rates (min total >=", MIN_TOTAL, "):
", highest[["arrest", "no_arrest", "total", "arrest_rate"]])
print("
Top 10 lowest arrest rates (min total >=", MIN_TOTAL, "):
", lowest[["arrest", "no_arrest", "total", "arrest_rate"]])
